# Build a Knowledge Graph for Your Firewall Policy — the guided version

*A warm, step-by-step lab for network and security engineers. You bring the firewall.*
*We bring the graph.*

A router asks *"which way?"*. A firewall asks a blunter question: *"do I let this*
*through — yes or no?"* It does not forward packets toward a destination; it **permits**
or **denies** a flow. And here is the thing every application quietly depends on: a real
application path is a **chain of flows** — web tier to app tier, app tier to database —
and each hop crosses a **zone boundary** and a **firewall rule**. The whole chain is only
as open as its **most-restrictive hop**. One `deny` anywhere, or one firewall that drops,
and the app dies — even though every *other* rule on the path is perfectly fine.

No single rule table shows you that chain. Your perimeter firewall doesn't know what the
internal firewall is doing; neither of them knows which *application* dies when a rule
flips. A **knowledge graph** makes the whole path explicit — so you can see exactly which
application goes dark when a rule flips to `deny` or a firewall goes down.

By the end of this notebook you will have built, from an empty page, a small graph that
answers the question no `show access-list` can:

> *"Someone just changed the app-to-database rule to deny. Which application just broke?"*

and watched it print **`Checkout`** — a fact that lives in nobody's rule table.

### First, calm the nerves: this is **not** machine learning
No training. No model. No embeddings. No AI guessing whether a packet is malicious. A
knowledge graph is just **structured facts** (things, and the named links between them)
plus **queries** that walk those links. Everything is **deterministic and inspectable** —
run it twice, get the same answer, and you can point at every node that produced it. If
you can read a firewall rule base, you can read every line in this lab.

### What you need
Nothing. This runs in **Google Colab** with zero setup, using plain **Python +**
**networkx + matplotlib** (both already installed in Colab). No database, no Docker, no
credentials. Press *Runtime -> Run all*, or run each cell in order with **Shift+Enter**.


## The whole vocabulary — five words, each one a firewall thing you already know

Before any code, here is the *entire* mental model. Everything after this is these five
ideas, repeated. You don't have to learn what any of them *mean* — only which firewall
thing each one maps onto.

| Graph word | Plain meaning | The firewall thing it already is |
|---|---|---|
| **node** | a thing | a firewall, a zone, a rule, a flow, an application |
| **edge** | a named, directed link between two nodes | "this flow crosses that firewall", "this rule governs that flow" |
| **label** | a node's *type* | `Zone`, `Firewall`, `Rule`, `Flow`, `Service` |
| **property** | a fact stored *on* a node or edge | `state='down'`, `action='deny'`, `trust='low'` |
| **traversal** | walking edges to answer a question | following an app's flows across every firewall to see if any hop is shut |

That's it. Hold those five words and the rest is your day job wearing a new hat.

One idea deserves a spotlight, because it is the whole trick: **the permit/deny decision**
**is a fact that lives on an *edge*, not a node.** "This rule denies this flow" is not
really a property of the rule alone, nor of the flow alone — it is a property of the
*relationship between them*. You already think this way: a rule is meaningless until you
say *what* it permits and *what* it denies. In a graph we write that literally — the
`permit` or `deny` rides **on the edge** from the rule to the flow. Watch for that moment;
it is where a policy spreadsheet turns into something you can *query*.


## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in
memory. We will use a **`DiGraph`** — a *directed* graph, where every edge has a
direction, an arrow. That matters to us: `Checkout -DEPENDS_ON_FLOW-> web-to-app` ("the
app needs this flow") is a different statement from the reverse. Security is full of
directional truth — traffic flows *from* a zone *to* a zone, a rule acts *on* a flow — so
directed edges are the honest choice.

Run the next cell. It imports the tools and hands you an empty graph to fill.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Both libraries ship pre-installed in Colab -- nothing to pip install.
# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
G = nx.DiGraph()

print("Empty graph ready:", G)
print("Nodes:", G.number_of_nodes(), " Edges:", G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready: DiGraph with 0 nodes and 0 edges` and
`Nodes: 0  Edges: 0`. That empty `G` is your blank whiteboard. Every step from here just
adds nodes and edges to it.


## Step 1 — Zones: the tiers, as nodes

**Theory.** A firewall's whole worldview is built on **zones** — named groups of assets
that share a trust level and a policy. "Inside", "outside", "DMZ", "PCI" — you name them
so you can write rules *between* them instead of host by host. Zones are the first thing
worth writing into our graph, because every application path is really a journey *across*
zone boundaries, and every boundary is where a firewall gets a vote.

We'll model a classic three-tier application, one zone per tier:

- **DMZ** — the low-trust edge where the web tier lives, exposed to users.
- **App Tier** — the medium-trust middle where application logic runs.
- **DB Tier** — the high-trust core where the database lives.

Each zone becomes a **node** with a `Zone` **label** and a `trust` **property**. Notice we
are *not* dumping every host or subnet in here — a node per *zone*, not per IP. We store
the *shape* of the security design, and we'll come back to that principle by name later.


In [ ]:
# add_node(name, **properties). The name is a unique handle; label & trust are facts on it.
G.add_node("DMZ",      label="Zone", trust="low")
G.add_node("App Tier", label="Zone", trust="medium")
G.add_node("DB Tier",  label="Zone", trust="high")

print(G.number_of_nodes(), "zones so far:", list(G.nodes))
print("Facts on DB Tier:", G.nodes["DB Tier"])

**Checkpoint.** Three nodes: `DMZ`, `App Tier`, `DB Tier`, and the facts on DB Tier show
`trust='high'`. You just wrote your zone model into a graph — no edges yet, just the
actors.


## Step 2 — Firewalls: the enforcers, each with a state

**Theory.** Between those zones sit the **firewalls** — the boxes that actually enforce
policy at each boundary. A firewall has one blunt property we care about above all: is it
**up** (enforcing normally) or **down** (failed)? And here is the security-specific twist
that a routing engineer must un-learn: when a firewall fails, traffic does **not** find
another way around. A well-designed firewall **fails closed** — if it drops, the flows
through it are *denied*, not rerouted. Down means shut. (One caveat: some inline/IPS or
hardware-bypass devices are explicitly built to fail **open** for availability — this lab
assumes the routed-firewall fail-closed default, which is the common enterprise case.)

We add two firewalls, mirroring the two boundaries the app must cross:

- **fw-perimeter** — guards the DMZ / App Tier boundary (the web-to-app hop).
- **fw-internal** — guards the App Tier / DB Tier boundary (the app-to-database hop).

Both start `state='up'`. The `state` is a **property** on the firewall node — because
"the firewall is down" is a fact about the box itself, unlike the permit/deny decision,
which (as we'll see) is a fact about a *relationship*.


In [ ]:
G.add_node("fw-perimeter", label="Firewall", state="up")   # DMZ  <-> App Tier
G.add_node("fw-internal",  label="Firewall", state="up")   # App Tier <-> DB Tier

for n, d in G.nodes(data=True):
    if d.get("label") == "Firewall":
        print(f'{n}: state={d["state"]}')

**Checkpoint.** Two firewalls print, both `state=up`: `fw-perimeter` and `fw-internal`.
They're floating unconnected for now — the next step introduces the *flows* that must
cross them, and that is where the application path starts to take shape.


## Step 3 — Flows: the application path is a chain from zone to zone across a firewall

**Theory.** Now the central object. A **flow** is a *required* piece of application
traffic — "the web tier must reach the app tier on 8080", "the app tier must reach the
database on 5432". It is not a packet; it is a *semantic requirement* the design promises
to keep open. Each flow has three facts that make it a path:

- where it starts (`FROM_ZONE`),
- where it ends (`TO_ZONE`),
- and which firewall it must **cross** to get there (`CROSSES`).

So each flow becomes a **node** with a `Flow` label, wired by three **edges** to the two
zones it spans and the one firewall it traverses. Read `app-to-db -CROSSES-> fw-internal`
literally: *"this flow only exists if it can get through fw-internal."*

This is the chain the intro promised. The **Checkout** application (coming soon) needs
*both* flows — web-to-app **and** app-to-db — and a chain is only as open as its
most-restrictive link. Model the flows and the chain becomes something you can walk.


In [ ]:
G.add_node("web->app", label="Flow", port=8080)   # web tier reaches app tier
G.add_node("app->db",  label="Flow", port=5432)   # app tier reaches database

# each flow spans two zones ...
G.add_edge("web->app", "DMZ",      rel="FROM_ZONE")
G.add_edge("web->app", "App Tier", rel="TO_ZONE")
G.add_edge("app->db",  "App Tier", rel="FROM_ZONE")
G.add_edge("app->db",  "DB Tier",  rel="TO_ZONE")

# ... and each must cross exactly one firewall to get there
G.add_edge("web->app", "fw-perimeter", rel="CROSSES")
G.add_edge("app->db",  "fw-internal",  rel="CROSSES")

for flow in ["web->app", "app->db"]:
    hop = {d["rel"]: v for _, v, d in G.out_edges(flow, data=True)}
    print(f'{flow}:  {hop["FROM_ZONE"]} -> {hop["TO_ZONE"]}  across {hop["CROSSES"]}')

**Checkpoint.** Two flows print their whole path:

```
web->app:  DMZ -> App Tier  across fw-perimeter
app->db:  App Tier -> DB Tier  across fw-internal
```

You've laid down the application path as a chain of flows, each pinned to the firewall it
depends on. There's still no permit or deny anywhere — that's the pivotal step, two steps
away. First, let's look at what we have.


## See it — draw the graph

**Theory.** Text is great during an incident, but to *understand* a design you want the
picture. We'll colour nodes by their **label** (zones one colour, firewalls another,
flows another) so the tiers and boundaries jump out. This is the same instinct as a
network diagram — except this diagram is generated *from the data*, so it can never drift
out of sync with the truth. (We curve the arrows slightly so any two-way pair stays
readable.)


In [ ]:
def draw(G, title="Firewall knowledge graph"):
    colors = {"Zone": "#0f7f74", "Firewall": "#4a5568", "Rule": "#8a6fb0",
              "Flow": "#3aa0ff", "Service": "#c8702a"}
    node_colors = [colors.get(G.nodes[n].get("label"), "#cccccc") for n in G]
    pos = nx.spring_layout(G, seed=7)   # fixed seed => stable, repeatable layout

    plt.figure(figsize=(11, 7.5))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1700, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=8)

    # a denied flow is the incident -- draw those edges dashed red so they stand out
    deny  = [(u, v) for u, v, d in G.edges(data=True) if d.get("action") == "deny"]
    other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in deny]
    nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=14,
                           edge_color="#8894a0", connectionstyle="arc3,rad=0.12")
    nx.draw_networkx_edges(G, pos, edgelist=deny, arrows=True, arrowsize=14,
                           edge_color="#d34b3f", width=2.0, style="dashed",
                           connectionstyle="arc3,rad=0.12")

    labels = {}
    for u, v, d in G.edges(data=True):
        labels[(u, v)] = f'GOVERNS:{d["action"]}' if d.get("rel") == "GOVERNS" else d.get("rel", "")
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=6)

    plt.axis("off"); plt.title(title); plt.tight_layout(); plt.show()

draw(G)

**Reading the picture.** You should see the zones (teal), the firewalls (slate), and the
two flows (blue), wired with `FROM_ZONE` / `TO_ZONE` / `CROSSES` arrows. This is still
just a topology — a prettier version of what you could draw on a whiteboard. It becomes a
*knowledge* graph in the next step, when we add the thing a diagram has never carried: the
**permit/deny decision itself**, welded to the exact flow it governs.


## Step 4 — Rules and the permit/deny edge (this is the pivotal step)

**Theory.** Here is the idea the whole lab pivots on. A **rule** by itself is inert — it
means nothing until you say *what it acts on* and *whether it permits or denies*. So the
`permit`/`deny` verdict does not belong on the rule node, and it doesn't belong on the
flow node. It belongs on the **edge between them** — the `GOVERNS` relationship. Read
`rule-app-db -GOVERNS(deny)-> app->db` literally: *"this rule governs that flow, and its
verdict is deny."*

We also connect each firewall to the rules it enforces with an `ENFORCES` edge — because a
rule lives *on* a firewall, and that's how a firewall failing takes its rules down with it.

The wiring, and the incident we're deliberately baking in:

- `fw-perimeter -ENFORCES-> rule-web-app`, and `rule-web-app -GOVERNS(permit)-> web->app`.
  The web-to-app hop is open. Good.
- `fw-internal -ENFORCES-> rule-app-db`, and `rule-app-db -GOVERNS(deny)-> app->db`.
  **This is the incident** — someone tightened the internal rule to `deny`. One edge, one
  word, and the app-to-database hop is shut.

That single `deny`, sitting on one edge, is the 3 AM change ticket — now a fact you can
query against.

*(One honest simplification: real firewalls and ACLs evaluate rules **top-down,**
**first-match-wins** — a permit above a deny shadows it. To keep this model clear we assume
**one verdict per flow** — a deny blocks it — rather than modeling rule order. That's fair
here because each flow carries a single governing rule; adding rule-order is a natural next
step once the shape clicks.)*


In [ ]:
G.add_node("rule-web-app", label="Rule")
G.add_node("rule-app-db",  label="Rule")

# each rule lives on (is enforced by) a firewall
G.add_edge("fw-perimeter", "rule-web-app", rel="ENFORCES")
G.add_edge("fw-internal",  "rule-app-db",  rel="ENFORCES")

# the verdict is a property ON the GOVERNS edge, not on either node
G.add_edge("rule-web-app", "web->app", rel="GOVERNS", action="permit")
G.add_edge("rule-app-db",  "app->db",  rel="GOVERNS", action="deny")   # <-- the incident

for r, f, d in G.edges(data=True):
    if d.get("rel") == "GOVERNS":
        print(f'{r} --GOVERNS({d["action"]})--> {f}')

**Checkpoint.** Two `GOVERNS` edges print, and exactly one reads `deny`:

```
rule-web-app --GOVERNS(permit)--> web->app
rule-app-db --GOVERNS(deny)--> app->db
```

That single `deny` on the `app->db` edge is the whole incident, sitting in your graph as a
fact. It isn't a *problem* yet, though — the graph doesn't know anything *needs* that flow.
That's the final, missing node.


## Step 5 — The application: the node the rule base never had

**Theory.** This is the node your firewalls were never designed to hold, and the reason
the whole lab exists. A firewall knows zones, rules, flows, ports. It has never once known
that the web-to-app **and** app-to-db flows together are what keep **Checkout** alive, and
that when either one is denied, **customers can't pay**.

So we add a **Service** node, `Checkout`, with a `criticality` property, and we wire it to
the flows it needs with `DEPENDS_ON_FLOW` edges — *"this application depends on that
flow."* Checkout depends on **both** flows, which is exactly what makes it a chain: if
*either* hop is shut, Checkout is down. No `show` command holds this link between a rule
and a revenue-earning app. It has lived only in tribal knowledge. You're about to make it a
first-class, walkable fact.


In [ ]:
G.add_node("Checkout", label="Service", criticality="critical")
G.add_edge("Checkout", "web->app", rel="DEPENDS_ON_FLOW")
G.add_edge("Checkout", "app->db",  rel="DEPENDS_ON_FLOW")

print("Graph now has", G.number_of_nodes(), "nodes and", G.number_of_edges(), "edges.")
print("Checkout depends on flows:",
      [v for _, v, d in G.out_edges("Checkout", data=True) if d.get("rel") == "DEPENDS_ON_FLOW"])
draw(G, title="Firewall knowledge graph -- now with the application")

**Checkpoint.** The graph has grown to **10 nodes and 12 edges**, including an orange
`Service` node, `Checkout`, joined by two `DEPENDS_ON_FLOW` edges to the flows. In the
redrawn picture the denied hop is now a **dashed red** arrow, and you can *trace with your
finger*: Checkout -> app-to-db -> the red deny. In the next step we make the computer trace
it for you.


## Step 6 — The 3 AM question: blast radius as a traversal

**Theory.** Everything so far was setup for this. A **traversal** is just walking edges to
answer a question. Our walk answers:

> *"For each application, is any flow it depends on currently blocked?"*

And a flow is **blocked** — in firewall logic — when *any* of these is true:

1. a firewall it `CROSSES` is **down** (fail-closed: down means shut); **or**
2. a rule that `GOVERNS` it says **deny**; **or**
3. **no** rule permits it at all (the default-deny that every firewall ends with).

The walk: for each `Service`, follow its `DEPENDS_ON_FLOW` edges to each flow, then check
that flow's `CROSSES` firewalls and its `GOVERNS` rules. If any hop is blocked, the app is
at risk — because the chain is only as open as its most-restrictive link.

Crucially, the query is **conditioned on state and action** — flip the deny back to permit
and step 2 finds nothing, so the app clears. That responsiveness is the difference between
a static diagram and a live model. Run it.


In [ ]:
def flow_block_reason(G, flow):
    """Why is this flow blocked, if it is? Returns a reason string, or None if open."""
    # 1) any firewall on the flow's path that is down (fail-closed)
    for _, fw, e in G.out_edges(flow, data=True):
        if e.get("rel") == "CROSSES" and G.nodes[fw].get("state") == "down":
            return f"firewall {fw} is down"
    # 2) / 3) the rules that govern this flow -- collect permits and denies
    permits, denies = [], []
    for rule, _, e in G.in_edges(flow, data=True):
        if e.get("rel") == "GOVERNS":
            (denies if e.get("action") == "deny" else permits).append(rule)
    if denies:
        return f"rule {denies[0]} denies it"
    if not permits:
        return "no permit rule (default deny)"
    return None   # open: firewalls up, a permit exists, no deny

def blast_radius(G):
    """Applications put at risk because a flow they depend on is blocked."""
    at_risk = []
    for svc, d in G.nodes(data=True):
        if d.get("label") != "Service":
            continue
        for _, flow, e in G.out_edges(svc, data=True):
            if e.get("rel") != "DEPENDS_ON_FLOW":
                continue
            reason = flow_block_reason(G, flow)
            if reason:
                at_risk.append((svc, flow, reason))
    return at_risk

hits = blast_radius(G)
for svc, flow, reason in hits:
    print(f"{svc}  needs  {flow}  ->  BLOCKED ({reason})  ->  AT RISK")
if not hits:
    print("nothing at risk")

The change ticket only ever said *"tightened rule on fw-internal"* — one line, no
mention of any application. Your graph just told you the **Checkout app is at risk**, and
showed its work: the exact flow that's blocked and *why*. You got that answer because
*you* added the one node the rule base never had — the application — and wired it to the
flows it depends on. That is the entire pitch of a knowledge graph, and you built it from
an empty page.


## Step 7 — What-if: fix the rule, break it again, then drop a firewall

**Theory.** Because both levers — the rule's `action` and the firewall's `state` — are
plain properties, you can *change* them and re-ask, running "what if?" on a **model**,
safely, with no change window and nobody paged. Flip the deny back to `permit`: Checkout
clears. Flip it to `deny` again: Checkout returns. Then leave the rule healthy but drop
`fw-internal` — and Checkout breaks anyway, this time because the firewall fails closed.
Two different root causes, one blast radius. This is the humble beginning of pre-incident
planning: ask the graph what *would* break before it does.


In [ ]:
# The verdict lives on the GOVERNS edge -- read and write it by [source][target].
G["rule-app-db"]["app->db"]["action"] = "permit"   # correct the deny
print("After permit:  ", sorted({s for s, _, _ in blast_radius(G)}) or "nothing at risk")

G["rule-app-db"]["app->db"]["action"] = "deny"     # break it again
print("After re-deny: ", sorted({s for s, _, _ in blast_radius(G)}))

# Now the OTHER failure mode: rule is fine, but the firewall itself drops (fail-closed).
G["rule-app-db"]["app->db"]["action"] = "permit"   # rule healthy again
G.nodes["fw-internal"]["state"] = "down"             # but the box goes down
print("Firewall down: ", sorted({s for s, _, _ in blast_radius(G)}))

G.nodes["fw-internal"]["state"] = "up"               # restore the firewall
G["rule-app-db"]["app->db"]["action"] = "deny"      # back to the baked-in incident

**Checkpoint.** You should see:

```
After permit:   nothing at risk
After re-deny:  ['Checkout']
Firewall down:  ['Checkout']
```

Same graph, same query — only the `action` on one edge, then the `state` on one node,
changed. The answer *responded* both times, and told you Checkout breaks whether the rule
denies **or** the firewall drops. That is what makes it a model rather than a drawing. (We
left the graph back at the baked-in `deny` so later cells start where we expect.)


## Your turn #1 — a second application on the same flow

Real flows rarely serve just one app. Suppose a **Reporting** service also rides on the
`app->db` flow (it reads the same database). Add it, wire it with a `DEPENDS_ON_FLOW`
edge, and re-run `blast_radius`. Because that flow is still denied, you should now see
**two** applications surface from the one rule change — because one edge of extra truth is
all it takes.

Fill in the cell below (there's a `# TODO`), then run it. The solution follows if you want
to check.


In [ ]:
# TODO: add a "Reporting" Service node (criticality="high"),
#       then a DEPENDS_ON_FLOW edge from "Reporting" to "app->db".

# G.add_node(...)
# G.add_edge(...)

for svc, flow, reason in blast_radius(G):
    print(f"AT RISK: {svc}  (flow {flow} blocked: {reason})")

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node("Reporting", label="Service", criticality="high")
G.add_edge("Reporting", "app->db", rel="DEPENDS_ON_FLOW")

for svc, flow, reason in blast_radius(G):
    print(f"AT RISK: {svc}  (flow {flow} blocked: {reason})")
```

Now both `Checkout` **and** `Reporting` come back from the one denied flow. The blast
radius grew the instant you told the graph one more true thing — you didn't touch the
query at all.
</details>


In [ ]:
# (Solution, applied -- so the rest of the notebook has both apps in the graph.)
G.add_node("Reporting", label="Service", criticality="high")
G.add_edge("Reporting", "app->db", rel="DEPENDS_ON_FLOW")
print("Applications at risk now:", sorted({s for s, _, _ in blast_radius(G)}))

## Quiet reveal — you have been writing an *ontology*

Time to name the thing you've been doing. Every step, you made two very different kinds of
decision, and it's worth seeing the seam between them:

- *"A `Flow` goes `FROM_ZONE` a `Zone` and `CROSSES` a `Firewall`. A `Rule` `GOVERNS` a
  `Flow` with an action. A `Service` `DEPENDS_ON_FLOW` a `Flow`."* — these are the
  **rules**: which node types exist, which edges are allowed, what shape a valid fact
  takes. That is the **schema**. Its fancy name is an **ontology** — and here's the
  friendly definition: *an ontology is the RFC of your graph, the agreed vocabulary
  written down as structure.* It plays the same role the firewall's data model plays: it
  says what a valid policy statement even *is*.

- *"This particular firewall is `fw-internal`, and its rule `rule-app-db` denies the
  `app->db` flow."* — these are the **instances**: the actual data.

A rule of thumb that keeps the two straight forever: **if it has a hostname, an IP, a
rule ID, or an app name, it is data (an instance), never schema.** `Firewall` is schema;
`fw-internal` is data. `GOVERNS` is schema; "this rule denies app-to-db" is data. Keep the
vocabulary small and stable; let the instances be many. That single discipline is the
difference between a graph you can grow for years and a swamp you abandon in a month.


## A peek at the real thing (1/2) — the same policy in Neo4j + Cypher

We used networkx so you could run everything with zero setup. But the *shapes* you built
are exactly what you'd build in a real graph database like **Neo4j**, whose query language
is **Cypher**. Cypher is worth a glance because it reads almost like the arrows we've been
drawing — `(node)-[:EDGE]->(node)`.

Here is the firewall, its rule, and the denied flow as Cypher. `MERGE` means "find this
node/edge or create it if missing" (so re-running is safe); `SET` assigns properties. This
is *reference only* — you don't run it here:

```cypher
MERGE (fw:Firewall {id: 'fw-internal'})
SET   fw.state = 'up';

MERGE (rule:FirewallRule {id: 'rule-app-db'});
MERGE (flow:TrafficFlow {id: 'app->db'})
SET   flow.port = 5432;

MERGE (fw)-[:ENFORCES]->(rule);

// the verdict, as a property on the relationship -- same as our edge
MERGE (rule)-[g:GOVERNS]->(flow)
SET   g.action = 'deny';
```

See it? `(rule)-[g:GOVERNS]->(flow) SET g.action='deny'` is the same statement as our
`G.add_edge("rule-app-db", "app->db", rel="GOVERNS", action="deny")`. Same nodes, same
edge, same verdict-on-the-edge — one just happens to live in a database that scales to
millions of rules.


## A peek at the real thing (2/2) — the 3 AM question in Cypher

Our `blast_radius` walk is a Python function with a couple of loops. In Cypher, the
"denied-rule" half of that traversal is a few lines, because in a graph database you
**draw the shape you're looking for** and let the engine find every match — no manual
loops:

```cypher
MATCH (svc:Service)-[:DEPENDS_ON_FLOW]->(flow:TrafficFlow)
MATCH (rule:FirewallRule)-[g:GOVERNS {action:'deny'}]->(flow)
RETURN svc.id AS app, flow.id AS blocked_flow,
       collect(DISTINCT rule.id) AS denying_rules;
```

Read the second line as a sentence: *"match a rule whose GOVERNS edge to this flow is a
deny."* That's the same step you wrote in Python — the pattern you `MATCH` **is** the
traversal. Run it against the real firewall dataset in the companion Neo4j lab and
`Checkout` comes back with `app->db` among the blocked flows. (A production query would
also `OPTIONAL MATCH` the firewall's `state` to catch the fail-closed case, exactly like
our `flow_block_reason` checks both.) Different engine; identical thinking. If you can read
the Python above, you can read the Cypher — you already speak this language.


## Your turn #2 — make the query respond to a *different* failure

So far the failure has always been a `deny`. Let's prove the model reacts to a firewall
**going down**, too — the fail-closed case. We'll build a *healthy* path for a new app, so
it starts safe, then break the firewall under it:

1. add a **Cache Tier** zone and a firewall `fw-cache` (`state='up'`);
2. add a flow `app->cache` (`FROM_ZONE` App Tier, `TO_ZONE` Cache Tier, `CROSSES`
   fw-cache), a rule `rule-app-cache` that `GOVERNS` it with `action='permit'`, and a
   **Search API** service that `DEPENDS_ON_FLOW` it;
3. re-run `blast_radius` — Search API should **not** appear (its path is permitted and up);
4. now set `fw-cache` `state='down'` and re-run — Search API **appears**.

This is Step 7 in your own hands, via the *other* lever. Fill in the `# TODO`s, then run.


In [ ]:
# TODO 1: build a healthy path for a new app:
#   - Zone "Cache Tier" (trust="medium");  Firewall "fw-cache" (state="up")
#   - Flow "app->cache": FROM_ZONE "App Tier", TO_ZONE "Cache Tier", CROSSES "fw-cache"
#   - Rule "rule-app-cache": fw-cache ENFORCES it; it GOVERNS "app->cache" action="permit"
#   - Service "Search API" (criticality="medium") that DEPENDS_ON_FLOW "app->cache"

print("With fw-cache UP:  ", sorted({s for s, _, _ in blast_radius(G)}))

# TODO 2: take the firewall down (G.nodes["fw-cache"]["state"] = "down")
#         and print blast_radius again to watch Search API appear.

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node("Cache Tier", label="Zone", trust="medium")
G.add_node("fw-cache",   label="Firewall", state="up")
G.add_node("app->cache", label="Flow", port=6379)
G.add_edge("app->cache", "App Tier",   rel="FROM_ZONE")
G.add_edge("app->cache", "Cache Tier", rel="TO_ZONE")
G.add_edge("app->cache", "fw-cache",   rel="CROSSES")
G.add_node("rule-app-cache", label="Rule")
G.add_edge("fw-cache", "rule-app-cache", rel="ENFORCES")
G.add_edge("rule-app-cache", "app->cache", rel="GOVERNS", action="permit")
G.add_node("Search API", label="Service", criticality="medium")
G.add_edge("Search API", "app->cache", rel="DEPENDS_ON_FLOW")

print("With fw-cache UP:  ", sorted({s for s, _, _ in blast_radius(G)}))
# -> Checkout, Reporting   (Search API is safe: permitted and up)

G.nodes["fw-cache"]["state"] = "down"
print("With fw-cache DOWN:", sorted({s for s, _, _ in blast_radius(G)}))
# -> now Search API joins the list
```

Before you broke the firewall, Search API was *in the graph* but *not at risk* — the query
correctly ignored a healthy path. The moment `fw-cache` went down, the exact same query
surfaced it. You didn't teach the query about caches; you told the graph the truth, and the
traversal did the rest.
</details>


In [ ]:
# (Solution, applied -- leaves fw-cache healthy again so later cells start clean.)
G.add_node("Cache Tier", label="Zone", trust="medium")
G.add_node("fw-cache",   label="Firewall", state="up")
G.add_node("app->cache", label="Flow", port=6379)
G.add_edge("app->cache", "App Tier",   rel="FROM_ZONE")
G.add_edge("app->cache", "Cache Tier", rel="TO_ZONE")
G.add_edge("app->cache", "fw-cache",   rel="CROSSES")
G.add_node("rule-app-cache", label="Rule")
G.add_edge("fw-cache", "rule-app-cache", rel="ENFORCES")
G.add_edge("rule-app-cache", "app->cache", rel="GOVERNS", action="permit")
G.add_node("Search API", label="Service", criticality="medium")
G.add_edge("Search API", "app->cache", rel="DEPENDS_ON_FLOW")

G.nodes["fw-cache"]["state"] = "down"
print("fw-cache down, at risk:", sorted({s for s, _, _ in blast_radius(G)}))
G.nodes["fw-cache"]["state"] = "up"     # restore health
print("restored, at risk:    ", sorted({s for s, _, _ in blast_radius(G)}))

## Make it yours — swap in *your* application

Now the moment it becomes your tool, not mine. Change the values below to **one**
application, **one** flow it needs, **one** firewall that flow crosses, and the two zones
it spans — from a design *you* actually run. We add a rule that **denies** that flow for
you (the modelled incident), so your application shows up. Run it, and watch **your own
app's name** come back from `blast_radius` — proof that the machine you built understands
your network, not just the demo's.

Keep it to the smallest slice that answers one real question. You can always add one more
node tomorrow — that's the whole promise of a graph you can grow.


In [ ]:
# --- change these values to your environment ---
MY_APP    = "Billing"
MY_FLOW   = "billing->ledger"
MY_FW     = "fw-billing"
MY_SRC    = "App Tier"        # a zone that already exists, or add your own
MY_DST    = "Ledger Tier"
# ------------------------------------------------

# add the source zone only if it's new (don't clobber an existing zone's facts)
if MY_SRC not in G:
    G.add_node(MY_SRC, label="Zone", trust="medium")
G.add_node(MY_DST, label="Zone", trust="high")
G.add_node(MY_FW,  label="Firewall", state="up")
G.add_node(MY_FLOW, label="Flow")
G.add_edge(MY_FLOW, MY_SRC, rel="FROM_ZONE")
G.add_edge(MY_FLOW, MY_DST, rel="TO_ZONE")
G.add_edge(MY_FLOW, MY_FW,  rel="CROSSES")

MY_RULE = "rule-" + MY_FLOW
G.add_node(MY_RULE, label="Rule")
G.add_edge(MY_FW, MY_RULE, rel="ENFORCES")
G.add_edge(MY_RULE, MY_FLOW, rel="GOVERNS", action="deny")   # your modelled incident

G.add_node(MY_APP, label="Service", criticality="critical")
G.add_edge(MY_APP, MY_FLOW, rel="DEPENDS_ON_FLOW")

print(f"If the {MY_FLOW} rule flips to deny, these apps are at risk:")
for svc, flow, reason in blast_radius(G):
    if svc == MY_APP:
        print(f"  AT RISK: {svc}   (flow {flow} blocked: {reason})")

**Checkpoint.** Your own application name — `Billing`, or whatever you typed — prints as
*at risk*, with the blocked flow and the reason. That is the moment the tool stopped being
a tutorial and became yours. Every other application you run is just a few more lines
away.


## The one rule that keeps this from becoming a swamp

You'll be tempted to model everything. Don't. Here is the discipline, in one line:

> **Model the nouns of your security design review, not the packets on the wire.**

Zones, firewalls, rules, required flows, applications — the **nouns** you'd draw on a
whiteboard while arguing about a policy change — those belong in the graph. Individual
connection logs, per-packet allow/drop counters, the full 5,000-line rule base with every
shadowed and redundant entry, NetFlow records — those are the **firehose**; leave them in
the SIEM and the firewall management console that already store them well, and let the
graph *reference* them when it needs to. The graph holds the *shape of the dependency* —
which app needs which flow across which firewall; your logging stack holds the volume.
Keep that line sharp and your graph stays queryable for years.


## You built a knowledge graph

Look back at the distance. You started with an empty page and five plain words. You added
zones, firewalls, and flows — a topology. Then you added the permit/deny verdict on an
edge and the one node the rule base never had, the **application** — and the topology
became *knowledge*. Finally you asked it the question no rule table can answer, and it
printed **Checkout**, then responded correctly every time you flipped a rule or dropped a
firewall underneath it.

The important idea was never firewalls, and never networkx. It's this: **a security**
**design has a shape** — applications depend on flows, flows cross firewalls, rules govern
flows with a verdict — and once that shape is a graph, impact analysis stops being tribal
knowledge and becomes a traversal. A rule base tells you *what is allowed*. A graph you can
query tells you *what breaks* — the thing no single rule table ever shows you.

### Where to go next
- **The companion Neo4j lab** does this exact thing in a real graph database — same nodes,
  same edges, same 3 AM question — so the two Cypher peeks above become things you run.
- **Add NAT and ports.** The shapes generalize: a NAT policy is a node that `TRANSLATES` a
  flow's address; a port is a node a flow rides on. Same five words carry all of it.
- **Add the change layer.** Model a `ChangeEvent` node linked to the rule it touched, and
  "what change denied this flow, and when?" becomes one more traversal.

Go model one real application on your network, and ask it what a single `deny` would break.
